In [ ]:
# Databricks notebook source
# tutorial_name: 02-Day8-Medallion-Guide-and-Student-Flow
# notebookName: 02-Day8-Medallion-Guide-and-Student-Flow
# COMMAND ----------

# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths (same as Days 5–8)
notepoint_rel = "hands-on/day-08/notebooks/02-Day8-Medallion-Guide-and-Student-Flow.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("OK: SOURCE_CSV")
except Exception as e:
    print("SOURCE_CSV:", e)
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("OK: SOURCE_JSON (optional)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


# Day 8 — Item 22: Medallion job — guide and student flow

This notebook does **not** write medallion tables. It sets **paths**, checks prerequisites, and explains **what to run and in what order**. The actual tasks are **03** (bronze), **04** (silver), **05** (gold); optional checks are in **06**.

## Student flow — item 22 (read this first)

Follow the steps **in order**. Skipping a step is the usual cause of failures (for example, silver before bronze exists).

| Step | Where | What you do |
|------|--------|-------------|
| **1** | This notebook (cluster) | Run the **paths / prerequisite** code cell below. Fix any red errors before continuing. |
| **2** | `03-Day8-Medallion-Bronze-Task.ipynb` (cluster) | Run **all cells top to bottom** once. Confirms CSV → **Bronze** Delta at `BRONZE_PATH`. |
| **3** | `04-Day8-Medallion-Silver-Task.ipynb` (cluster) | Run **all cells**. Builds **Silver** from Bronze. If Bronze is missing, this notebook will error — go back to step 2. |
| **4** | `05-Day8-Medallion-Gold-Task.ipynb` (cluster) | Run **all cells**. Builds **Gold** from Silver and appends a **metrics** row. |
| **5** | **Workflows → Jobs → Create job** | Create **three tasks**, each pointing to a **different** notebook file (see table below). **No** `pipeline_stage` widget is required. |
| **6** | Job **Run now** | Watch the DAG: **bronze** → **silver** → **gold**. |
| **7** | `06-Day8-Medallion-PostRun-Checks.ipynb` (cluster) | Run after a successful job (or after step 4) to print counts, Delta history, and optional **P_BASIC** comparison. |

**Job task wiring (reference)**

| Task name | Notebook file | Depends on |
|-----------|---------------|------------|
| `bronze` | `03-Day8-Medallion-Bronze-Task.ipynb` | — |
| `silver` | `04-Day8-Medallion-Silver-Task.ipynb` | `bronze` |
| `gold` | `05-Day8-Medallion-Gold-Task.ipynb` | `silver` |

In the job UI: add each task as **Notebook**, paste the workspace path to each file, set **Depends on** for silver and gold, then **Run now**.


## How the three tasks relate

Each task is a **separate notebook**. The **job** enforces order: silver never starts until bronze succeeds, and gold waits for silver.

```mermaid
flowchart LR
  bronze[Task: Bronze notebook] --> silver[Task: Silver notebook]
  silver --> gold[Task: Gold notebook]
```

**Data flow:** `2010-summary.csv` on ABFS → **Bronze** Delta (raw + `ingest_ts`) → **Silver** Delta (non-null keys + `silver_built_ts`) → **Gold** Delta (one row per destination + `total_flights`).


### If something goes wrong

| Problem | Likely fix |
|---------|------------|
| Silver or gold fails with **path not found** | Run **03** then **04** then **05** interactively once, or fix **Depends on** in the job. |
| **Permission** on `abfss` | Same storage setup as Day 5; check `STUDENT_ID` and cluster credentials. |
| Task **Pending** | Cluster policy or job compute not starting; check **Runs** → task details. |
| Counts look wrong | Bronze uses **overwrite** each time; re-run the full chain. |

**Idempotency:** Bronze, silver, and gold each **overwrite** their Delta path. Re-running a task replaces that layer only; downstream tasks may need a re-run to see new data.
